In [2]:
pip install catboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 
%matplotlib inline
import seaborn as sns 
import warnings
warnings.filterwarnings('ignore')

from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor

In [13]:
df = pd.read_csv('./data/stud.csv')

In [14]:
x = df.drop(columns=['math_score'], axis=1)
y = df['math_score']

In [15]:
num_features = x.select_dtypes(exclude="object").columns
cat_features = x.select_dtypes(include="object").columns

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder()

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", oh_transformer, cat_features),
        ("StandardScaler", numeric_transformer, num_features)
    ]
)

In [16]:
x = preprocessor.fit_transform(x)

In [17]:
x.shape

(1000, 19)

In [18]:
#seperate dataset into train and test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [20]:
# create an evaluation function
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2_val = r2_score(true, predicted)
    return mae, mse, rmse, r2_val

In [22]:
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "XGBRegressor": XGBRegressor(),
    "CatBoosting Regressor": CatBoostRegressor(verbose=False),
    "AdaBoost Regressor": AdaBoostRegressor()
}

model_list = []
r2_list = []

for i in range (len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train)
    
    # make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # evaluate train and test dataset
    model_train_mae, _, model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)
    model_test_mae, _, model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)
    
    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])
    
    print('model performance for training set')
    print(model_train_rmse)
    print(model_train_mae)
    print(model_train_r2)
    print('-------------------------------------------')
    print('model performance for training set')
    print(model_test_rmse)
    print(model_test_mae)
    print(model_test_r2)
    
    r2_list.append(model_test_r2)

Linear Regression
model performance for training set
5.323050852720514
4.266711846071957
0.8743172040139593
-------------------------------------------
model performance for training set
5.393993869732843
4.21476314247485
0.8804332983749565
Lasso
model performance for training set
6.593815587795566
5.206302661246526
0.8071462015863456
-------------------------------------------
model performance for training set
6.519694535667419
5.157881810347763
0.8253197323627853
Ridge
model performance for training set
5.323324922741654
4.264987823725981
0.8743042615212909
-------------------------------------------
model performance for training set
5.390387016935642
4.211100688014261
0.8805931485028737
K-Neighbors Regressor
model performance for training set
5.707683417990174
4.516749999999999
0.8554978341651085
-------------------------------------------
model performance for training set
7.253040741647602
5.621
0.7838129945787431
Decision Tree
model performance for training set
0.27950849718747

In [23]:
pd.DataFrame(list(zip(model_list, r2_list)), columns=['model_name', 'r2_value']).sort_values(by=["r2_value"], ascending=False)

,model_name,r2_value
2,Ridge,0.880593
0,Linear Regression,0.880433
7,CatBoosting Regressor,0.851632
5,Random Forest Regressor,0.850271
8,AdaBoost Regressor,0.844290
6,XGBRegressor,0.827797
1,Lasso,0.825320
3,K-Neighbors Regressor,0.783813
4,Decision Tree,0.741903
